# 07 — Deep Learning Regression

## Purpose

This notebook trains an end-to-end RGB image regression model for pasture biomass prediction using a frozen ImageNet-pretrained EfficientNetB0 backbone and a five-output linear regression head.

The experiment reuses the existing five-fold assignments created earlier in the project, trains one model per fold, saves the fitted Keras models, records fold histories, generates out-of-fold predictions, and produces the same style of integrity checks and summary reports used in the earlier notebooks.

## What this notebook adds

1. Direct image-to-target regression instead of feature extraction followed by tree-based modeling.
2. TensorFlow `tf.data` input pipelines with caching, prefetching, and parallel loading.
3. Mixed precision support when a GPU is available.
4. Fold-wise training diagnostics, prediction diagnostics, and artifact validation.
5. A comparable reporting layer so the deep-learning results can be reviewed alongside Notebooks 04 to 06.

## Experimental design

The workflow is intentionally conservative:

- reuse the fold assignments from Notebook 02;
- reuse the processed image paths saved in the fold table;
- keep the output targets exactly aligned with `TARGET_COLUMNS`;
- save all artifacts using the project's existing reporting and figure directories.

The goal is to produce a fully reproducible deep-learning baseline that fits naturally into the existing project structure.

In [1]:
import inspect
import json
import os
import platform
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT_CANDIDATE = Path.cwd().resolve()
while not (PROJECT_ROOT_CANDIDATE / "src").exists() and PROJECT_ROOT_CANDIDATE.parent != PROJECT_ROOT_CANDIDATE:
    PROJECT_ROOT_CANDIDATE = PROJECT_ROOT_CANDIDATE.parent

if str(PROJECT_ROOT_CANDIDATE) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT_CANDIDATE))

from src.config import (
    PROJECT_ROOT,
    DATA_DIR,
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    FOLDS_DIR,
    TRAIN_IMAGE_DIR,
    MODELS_DIR,
    REPORTS_DIR,
    PREDICTIONS_DIR,
    FIGURES_DIR,
    DEEP_LEARNING_MODEL_DIR,
    DEEP_LEARNING_FIGURES_DIR,
    IMAGE_SIZE,
    RANDOM_SEED,
    N_FOLDS,
    CNN_BATCH_SIZE,
    create_directories,
)
from src.modeling import TARGET_COLUMNS
from src.feature_extraction import load_and_preprocess_image
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

create_directories()

print("=" * 60)
print("NOTEBOOK 07 - DEEP LEARNING REGRESSION - ENVIRONMENT SETUP")
print("=" * 60)
print(f"Project Root            : {PROJECT_ROOT}")
print(f"Python Version           : {platform.python_version()}")
print(f"Random Seed              : {RANDOM_SEED}")
print(f"Number of Folds          : {N_FOLDS}")
print(f"Targets                  : {TARGET_COLUMNS}")
print(f"Image Size               : {IMAGE_SIZE}")
print("Imported shared helpers  : OK")
print("Imported shared constants: OK")


NOTEBOOK 07 - DEEP LEARNING REGRESSION - ENVIRONMENT SETUP
Project Root            : E:\DATAVIDWAN\Image2Biomass
Python Version           : 3.13.5
Random Seed              : 42
Number of Folds          : 5
Targets                  : ['Dry_Clover_g', 'Dry_Dead_g', 'Dry_Green_g', 'Dry_Total_g', 'GDM_g']
Image Size               : (224, 224)
Imported shared helpers  : OK
Imported shared constants: OK


In [2]:
import tensorflow as tf
from tensorflow.keras import applications, callbacks, layers, mixed_precision, models, optimizers

print("=" * 60)
print("TENSORFLOW / GPU CONFIGURATION")
print("=" * 60)
print(f"TensorFlow Version : {tf.__version__}")

physical_gpus = tf.config.list_physical_devices("GPU")
gpu_available = len(physical_gpus) > 0

if gpu_available:
    try:
        for gpu in physical_gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as exc:
        print(f"GPU memory growth configuration skipped: {exc}")

    mixed_precision.set_global_policy("mixed_float16")
    print("GPU Detected        : YES")
    print("Mixed Precision     : ENABLED")
else:
    mixed_precision.set_global_policy("float32")
    print("GPU Detected        : NO")
    print("Mixed Precision     : DISABLED")

print(f"Global Policy       : {mixed_precision.global_policy()}")

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.keras.utils.set_random_seed(RANDOM_SEED)

try:
    tf.config.experimental.enable_op_determinism()
    print("Deterministic ops   : ENABLED")
except Exception:
    print("Deterministic ops   : NOT AVAILABLE")

BATCH_SIZE = 8 if not gpu_available else max(16, CNN_BATCH_SIZE // 2)
AUTOTUNE = tf.data.AUTOTUNE
LEARNING_RATE = 1e-4
EPOCHS = 10
PATIENCE = 3
REDUCE_LR_PATIENCE = 2
MODEL_NAME = "EfficientNetB0Regression"
FEATURE_SET_NAME = "images_only"
CLIP_NEGATIVE_PREDICTIONS = True

print(f"Batch Size          : {BATCH_SIZE}")
print(f"Learning Rate       : {LEARNING_RATE}")
print(f"Epoch Budget        : {EPOCHS}")
print(f"Early Stopping      : {PATIENCE}")
print(f"Reduce LR Patience  : {REDUCE_LR_PATIENCE}")


TENSORFLOW / GPU CONFIGURATION
TensorFlow Version : 2.20.0
GPU Detected        : NO
Mixed Precision     : DISABLED
Global Policy       : <DTypePolicy "float32">
Deterministic ops   : ENABLED
Batch Size          : 8
Learning Rate       : 0.0001
Epoch Budget        : 10
Early Stopping      : 3
Reduce LR Patience  : 2


In [3]:
PROCESSED_IMAGE_DIR = PROCESSED_DATA_DIR / "images"
FOLD_ASSIGNMENTS_PATH = FOLDS_DIR / "fold_assignments.csv"
TRAIN_TARGETS_PATH = RAW_DATA_DIR / "train.csv"
CNN_REGRESSION_MODEL_DIR = MODELS_DIR / "cnn_regression"
CNN_REGRESSION_MODEL_DIR.mkdir(parents=True, exist_ok=True)
DEEP_LEARNING_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

required_paths = {
    FOLD_ASSIGNMENTS_PATH: "Notebook 02",
    TRAIN_TARGETS_PATH: "Project raw training labels",
    PROCESSED_IMAGE_DIR: "Notebook 02",
    CNN_REGRESSION_MODEL_DIR: "Notebook 07",
    DEEP_LEARNING_FIGURES_DIR: "Notebook 07",
    REPORTS_DIR: "Project reports directory",
    PREDICTIONS_DIR: "Project predictions directory",
}

print("=" * 60)
print("PATH VALIDATION")
print("=" * 60)
for path, source in required_paths.items():
    print(f"{path} <- {source}")
    if not path.exists():
        raise FileNotFoundError(f"Required path is missing: {path}")

print("All required paths exist.")


PATH VALIDATION
E:\DATAVIDWAN\Image2Biomass\data\processed\folds\fold_assignments.csv <- Notebook 02
E:\DATAVIDWAN\Image2Biomass\data\raw\train.csv <- Project raw training labels
E:\DATAVIDWAN\Image2Biomass\data\processed\images <- Notebook 02
E:\DATAVIDWAN\Image2Biomass\models\cnn_regression <- Notebook 07
E:\DATAVIDWAN\Image2Biomass\outputs\figures\deep_learning <- Notebook 07
E:\DATAVIDWAN\Image2Biomass\outputs\reports <- Project reports directory
E:\DATAVIDWAN\Image2Biomass\outputs\predictions <- Project predictions directory
All required paths exist.


In [4]:
fold_assignments = pd.read_csv(FOLD_ASSIGNMENTS_PATH)
train_targets = pd.read_csv(TRAIN_TARGETS_PATH)

print("=" * 60)
print("LOADED PROJECT ARTIFACTS")
print("=" * 60)
print(f"Fold assignments shape : {fold_assignments.shape}")
print(f"Training labels shape   : {train_targets.shape}")
print(f"Processed image dir     : {PROCESSED_IMAGE_DIR}")
print(f"Model dir               : {CNN_REGRESSION_MODEL_DIR}")
print(f"Figure dir              : {DEEP_LEARNING_FIGURES_DIR}")

fold_assignments.head()


LOADED PROJECT ARTIFACTS
Fold assignments shape : (357, 8)
Training labels shape   : (1785, 9)
Processed image dir     : E:\DATAVIDWAN\Image2Biomass\data\processed\images
Model dir               : E:\DATAVIDWAN\Image2Biomass\models\cnn_regression
Figure dir              : E:\DATAVIDWAN\Image2Biomass\outputs\figures\deep_learning


,image_path,processed_image_path,fold,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g
0,train/ID1011485656.jpg,e:\DATAVIDWAN\Image2Biomass\data\processed\ima...,3,0.0000,31.9984,16.2751,48.2735,16.2750
1,train/ID1012260530.jpg,e:\DATAVIDWAN\Image2Biomass\data\processed\ima...,4,0.0000,0.0000,7.6000,7.6000,7.6000
2,train/ID1025234388.jpg,e:\DATAVIDWAN\Image2Biomass\data\processed\ima...,1,6.0500,0.0000,0.0000,6.0500,6.0500
3,train/ID1028611175.jpg,e:\DATAVIDWAN\Image2Biomass\data\processed\ima...,3,0.0000,30.9703,24.2376,55.2079,24.2376
4,train/ID1035947949.jpg,e:\DATAVIDWAN\Image2Biomass\data\processed\ima...,0,0.4343,23.2239,10.5261,34.1844,10.9605


In [5]:
required_columns = ["image_path", "processed_image_path", "fold", *TARGET_COLUMNS]
missing_columns = [column for column in required_columns if column not in fold_assignments.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

train_df = fold_assignments.copy().reset_index(drop=True)
train_df["image_path"] = train_df["image_path"].astype(str)
train_df["processed_image_path"] = train_df["processed_image_path"].astype(str)

sample_count = len(train_df)
unique_images = train_df["image_path"].nunique()
fold_values = sorted(train_df["fold"].unique().tolist())
duplicate_image_count = int(train_df["image_path"].duplicated().sum())
missing_image_paths = [path for path in train_df["processed_image_path"] if not Path(path).exists()]

numeric_targets = train_df[TARGET_COLUMNS].to_numpy(dtype=np.float32)
train_nan_count = int(np.isnan(numeric_targets).sum())
train_inf_count = int(np.isinf(numeric_targets).sum())

print("=" * 60)
print("FOLD TABLE VALIDATION")
print("=" * 60)
print(f"Sample Count            : {sample_count}")
print(f"Unique Images           : {unique_images}")
print(f"Fold Values             : {fold_values}")
print(f"Duplicate Image Paths    : {duplicate_image_count}")
print(f"Missing Processed Images : {len(missing_image_paths)}")
print(f"Target NaN Count         : {train_nan_count}")
print(f"Target Inf Count         : {train_inf_count}")

assert sample_count == unique_images, "Fold table should contain one row per unique image."
assert sample_count > 0, "Training table is empty."
assert len(fold_values) == N_FOLDS, f"Expected {N_FOLDS} folds, found {len(fold_values)}."
assert set(fold_values) == set(range(N_FOLDS)), f"Expected fold labels 0..{N_FOLDS - 1}, found {fold_values}."
assert duplicate_image_count == 0, "Duplicate image paths detected."
assert len(missing_image_paths) == 0, "Some processed images are missing."
assert train_nan_count == 0, "Training targets contain NaN values."
assert train_inf_count == 0, "Training targets contain infinite values."

fold_counts = train_df["fold"].value_counts().sort_index()
fold_summary = fold_counts.rename_axis("fold").reset_index(name="count")
fold_summary


FOLD TABLE VALIDATION
Sample Count            : 357
Unique Images           : 357
Fold Values             : [0, 1, 2, 3, 4]
Duplicate Image Paths    : 0
Missing Processed Images : 0
Target NaN Count         : 0
Target Inf Count         : 0


,fold,count
0,0,72
1,1,72
2,2,71
3,3,71
4,4,71


## Data pipeline design

The notebook uses a compact TensorFlow data pipeline for each fold:

- image paths and target rows are sliced directly from the fold table;
- `load_and_preprocess_image` from `src.feature_extraction` performs decoding and resizing;
- caching, batching, and prefetching are applied with `AUTOTUNE`;
- augmentation layers live inside the model so they only affect training batches.

In [6]:
def make_dataset(image_paths, targets, training=False, batch_size=BATCH_SIZE):
    image_paths = [str(Path(path)) for path in image_paths]
    targets = np.asarray(targets, dtype=np.float32)

    dataset = tf.data.Dataset.from_tensor_slices((image_paths, targets))

    def _load_example(image_path, target_values):
        image = load_and_preprocess_image(image_path)
        return image, tf.cast(target_values, tf.float32)

    dataset = dataset.map(
        _load_example,
        num_parallel_calls=AUTOTUNE,
        deterministic=not training,
    )
    dataset = dataset.cache()

    if training:
        dataset = dataset.shuffle(
            buffer_size=min(len(image_paths), 1024),
            seed=RANDOM_SEED,
            reshuffle_each_iteration=True,
        )

    dataset = dataset.batch(batch_size, drop_remainder=False)
    dataset = dataset.prefetch(AUTOTUNE)

    return dataset


def build_augmentation_block():
    return models.Sequential(
        [
            layers.RandomFlip("horizontal"),
            layers.RandomRotation(0.08),
            layers.RandomContrast(0.12),
            layers.RandomZoom(0.08),
        ],
        name="augmentation",
    )


def build_regression_model():
    inputs = layers.Input(shape=(*IMAGE_SIZE, 3), name="image")
    x = layers.Rescaling(1.0 / 255.0, name="rescale")(inputs)
    x = build_augmentation_block()(x)

    efficientnet_kwargs = {
        "weights": "imagenet",
        "include_top": False,
        "input_shape": (*IMAGE_SIZE, 3),
    }
    if "include_preprocessing" in inspect.signature(applications.EfficientNetB0).parameters:
        efficientnet_kwargs["include_preprocessing"] = False

    backbone = applications.EfficientNetB0(**efficientnet_kwargs)
    backbone.trainable = False

    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
    x = layers.Dense(256, activation="relu", name="dense_256")(x)
    x = layers.Dropout(0.3, name="dropout")(x)
    outputs = layers.Dense(len(TARGET_COLUMNS), activation="linear", dtype="float32", name="regression_head")(x)

    model = models.Model(inputs=inputs, outputs=outputs, name="efficientnetb0_regression")
    model.compile(
        optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=tf.keras.losses.Huber(),
        metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")],
    )
    return model


print("TensorFlow data pipeline and model builder functions are ready.")


TensorFlow data pipeline and model builder functions are ready.


In [7]:
def compute_metrics_table(y_true, y_pred, target_names=TARGET_COLUMNS):
    records = []
    for target_index, target_name in enumerate(target_names):
        true_values = y_true[:, target_index]
        predicted_values = y_pred[:, target_index]
        records.append(
            {
                "target": target_name,
                "MAE": mean_absolute_error(true_values, predicted_values),
                "RMSE": float(np.sqrt(mean_squared_error(true_values, predicted_values))),
                "R2": r2_score(true_values, predicted_values),
            }
        )
    return pd.DataFrame(records)


def summarize_fold_metrics(fold_metrics):
    summary = (
        fold_metrics.groupby(["model", "feature_set", "target"], as_index=False)
        .agg(
            MAE_mean=("MAE", "mean"),
            MAE_std=("MAE", "std"),
            RMSE_mean=("RMSE", "mean"),
            RMSE_std=("RMSE", "std"),
            R2_mean=("R2", "mean"),
            R2_std=("R2", "std"),
        )
        .sort_values(["RMSE_mean", "MAE_mean"], ascending=[True, True])
        .reset_index(drop=True)
    )
    return summary


def build_model_ranking(cv_summary):
    ranking = (
        cv_summary.groupby(["model", "feature_set"], as_index=False)
        .agg(
            mean_MAE=("MAE_mean", "mean"),
            mean_RMSE=("RMSE_mean", "mean"),
            mean_R2=("R2_mean", "mean"),
        )
        .sort_values(["mean_RMSE", "mean_MAE"], ascending=[True, True])
        .reset_index(drop=True)
    )
    ranking.insert(0, "rank", np.arange(1, len(ranking) + 1))
    return ranking


def validate_prediction_array(predictions, expected_rows, expected_targets=len(TARGET_COLUMNS)):
    expected_shape = (expected_rows, expected_targets)
    if predictions.shape != expected_shape:
        raise ValueError(f"Expected prediction shape {expected_shape}, received {predictions.shape}.")

    nan_count = int(np.isnan(predictions).sum())
    inf_count = int(np.isinf(predictions).sum())
    if nan_count > 0:
        raise ValueError(f"Predictions contain {nan_count} NaN values.")
    if inf_count > 0:
        raise ValueError(f"Predictions contain {inf_count} infinite values.")

    return {
        "n_samples": predictions.shape[0],
        "n_targets": predictions.shape[1],
        "dtype": str(predictions.dtype),
        "nan_count": nan_count,
        "inf_count": inf_count,
        "prediction_min": float(np.min(predictions)),
        "prediction_max": float(np.max(predictions)),
        "prediction_mean": float(np.mean(predictions)),
        "prediction_std": float(np.std(predictions)),
    }


def build_prediction_frame(table, predictions, fold_value, target_names=TARGET_COLUMNS):
    prediction_df = pd.DataFrame(index=table.index)
    prediction_df["image_path"] = table["image_path"].values
    prediction_df["processed_image_path"] = table["processed_image_path"].values
    prediction_df["fold"] = fold_value
    for target_index, target_name in enumerate(target_names):
        prediction_df[f"actual_{target_name}"] = table[target_name].values
        prediction_df[f"pred_{target_name}"] = predictions[:, target_index]
        prediction_df[f"residual_{target_name}"] = table[target_name].values - predictions[:, target_index]
    return prediction_df


print("Metric and validation helpers are ready.")


Metric and validation helpers are ready.


In [8]:
def save_history_figure(history_frames, output_path):
    fig, axes = plt.subplots(len(history_frames), 2, figsize=(16, 4 * len(history_frames)), sharex=False)
    if len(history_frames) == 1:
        axes = np.expand_dims(axes, axis=0)

    for fold_index, (fold, history_df) in enumerate(history_frames.items()):
        loss_ax = axes[fold_index, 0]
        mae_ax = axes[fold_index, 1]

        if "loss" in history_df.columns:
            loss_ax.plot(history_df["loss"], label="train_loss", linewidth=2)
        if "val_loss" in history_df.columns:
            loss_ax.plot(history_df["val_loss"], label="val_loss", linewidth=2)
        loss_ax.set_title(f"Fold {fold} - Loss")
        loss_ax.set_xlabel("Epoch")
        loss_ax.set_ylabel("Huber Loss")
        loss_ax.legend()
        loss_ax.grid(True, alpha=0.3)

        if "mae" in history_df.columns:
            mae_ax.plot(history_df["mae"], label="train_mae", linewidth=2)
        if "val_mae" in history_df.columns:
            mae_ax.plot(history_df["val_mae"], label="val_mae", linewidth=2)
        mae_ax.set_title(f"Fold {fold} - MAE")
        mae_ax.set_xlabel("Epoch")
        mae_ax.set_ylabel("MAE")
        mae_ax.legend()
        mae_ax.grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)


def save_prediction_scatter(oof_df, output_path):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    for idx, target_name in enumerate(TARGET_COLUMNS):
        ax = axes[idx]
        actual = oof_df[f"actual_{target_name}"]
        predicted = oof_df[f"pred_{target_name}"]
        ax.scatter(actual, predicted, s=18, alpha=0.7)
        min_val = float(min(actual.min(), predicted.min()))
        max_val = float(max(actual.max(), predicted.max()))
        ax.plot([min_val, max_val], [min_val, max_val], linestyle="--", color="black", linewidth=1)
        ax.set_title(target_name)
        ax.set_xlabel("Actual")
        ax.set_ylabel("Predicted")
        ax.grid(True, alpha=0.25)
    axes[-1].axis("off")
    fig.suptitle("Actual vs Predicted OOF Values", y=1.02, fontsize=16)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)


def save_residual_plots(oof_df, output_path):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    for idx, target_name in enumerate(TARGET_COLUMNS):
        ax = axes[idx]
        residuals = oof_df[f"residual_{target_name}"]
        predicted = oof_df[f"pred_{target_name}"]
        ax.scatter(predicted, residuals, s=18, alpha=0.7)
        ax.axhline(0, color="black", linestyle="--", linewidth=1)
        ax.set_title(target_name)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Residual")
        ax.grid(True, alpha=0.25)
    axes[-1].axis("off")
    fig.suptitle("Residual Diagnostics", y=1.02, fontsize=16)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)


def save_residual_histogram(oof_df, output_path):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    for idx, target_name in enumerate(TARGET_COLUMNS):
        ax = axes[idx]
        residuals = oof_df[f"residual_{target_name}"]
        ax.hist(residuals, bins=30, alpha=0.8, color="#4C72B0", edgecolor="white")
        ax.axvline(0, color="black", linestyle="--", linewidth=1)
        ax.set_title(target_name)
        ax.set_xlabel("Residual")
        ax.set_ylabel("Count")
        ax.grid(True, alpha=0.25)
    axes[-1].axis("off")
    fig.suptitle("Residual Histogram", y=1.02, fontsize=16)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)


def save_prediction_distribution(oof_df, output_path):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    for idx, target_name in enumerate(TARGET_COLUMNS):
        ax = axes[idx]
        actual = oof_df[f"actual_{target_name}"]
        predicted = oof_df[f"pred_{target_name}"]
        ax.hist(actual, bins=24, alpha=0.55, label="Actual", density=True)
        ax.hist(predicted, bins=24, alpha=0.55, label="Predicted", density=True)
        ax.set_title(target_name)
        ax.set_xlabel("Value")
        ax.set_ylabel("Density")
        ax.legend()
        ax.grid(True, alpha=0.25)
    axes[-1].axis("off")
    fig.suptitle("Prediction Distribution Comparison", y=1.02, fontsize=16)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)


def save_target_barplot(metric_table, metric_name, output_path):
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(metric_table["target"], metric_table[metric_name], color="#4C72B0")
    ax.set_title(f"Per-target {metric_name}")
    ax.set_xlabel("Target")
    ax.set_ylabel(metric_name)
    ax.grid(True, axis="y", alpha=0.25)
    plt.xticks(rotation=20, ha="right")
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)


print("Plotting helpers are ready.")


Plotting helpers are ready.


## Cross-validation setup

Each fold trains the same EfficientNetB0 regression architecture on a different held-out validation split.

The notebook saves:

- a fitted `.keras` model for every fold;
- a history CSV for every fold;
- a fold-specific prediction CSV for every fold;
- aligned OOF predictions for the full dataset.

In [9]:
fold_histories = {}
fold_prediction_frames = []
fold_metrics_frames = []
oof_predictions = np.full((len(train_df), len(TARGET_COLUMNS)), np.nan, dtype=np.float32)
training_start = time.time()

print("=" * 60)
print("FOLD STORAGE INITIALIZED")
print("=" * 60)
print(f"OOF array shape : {oof_predictions.shape}")
print(f"Training rows   : {len(train_df)}")
print(f"Output model dir : {CNN_REGRESSION_MODEL_DIR}")
print(f"Output figure dir: {DEEP_LEARNING_FIGURES_DIR}")


FOLD STORAGE INITIALIZED
OOF array shape : (357, 5)
Training rows   : 357
Output model dir : E:\DATAVIDWAN\Image2Biomass\models\cnn_regression
Output figure dir: E:\DATAVIDWAN\Image2Biomass\outputs\figures\deep_learning


In [10]:
for fold in range(N_FOLDS):
    print("=" * 60)
    print(f"TRAINING FOLD {fold}")
    print("=" * 60)

    tf.keras.backend.clear_session()

    train_mask = train_df["fold"] != fold
    validation_mask = train_df["fold"] == fold
    train_rows = train_df.loc[train_mask].copy().reset_index(drop=True)
    validation_rows = train_df.loc[validation_mask].copy()

    X_train = train_rows["processed_image_path"].tolist()
    y_train = train_rows[TARGET_COLUMNS].to_numpy(dtype=np.float32)
    X_validation = validation_rows["processed_image_path"].tolist()
    y_validation = validation_rows[TARGET_COLUMNS].to_numpy(dtype=np.float32)

    train_dataset = make_dataset(X_train, y_train, training=True, batch_size=BATCH_SIZE)
    validation_dataset = make_dataset(X_validation, y_validation, training=False, batch_size=BATCH_SIZE)

    model = build_regression_model()

    model_path = CNN_REGRESSION_MODEL_DIR / f"fold{fold}.keras"
    history_path = CNN_REGRESSION_MODEL_DIR / f"fold{fold}_history.csv"
    checkpoint_path = CNN_REGRESSION_MODEL_DIR / f"fold{fold}_checkpoint.keras"
    prediction_path = PREDICTIONS_DIR / f"07_fold{fold}_oof_predictions.csv"

    fold_callbacks = [
        callbacks.EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=1,
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=REDUCE_LR_PATIENCE,
            min_lr=1e-6,
            verbose=1,
        ),
        callbacks.ModelCheckpoint(
            filepath=checkpoint_path,
            monitor="val_loss",
            mode="min",
            save_best_only=True,
            save_weights_only=False,
            verbose=1,
        ),
        callbacks.CSVLogger(str(history_path), append=False),
    ]

    history = model.fit(
        train_dataset,
        validation_data=validation_dataset,
        epochs=EPOCHS,
        callbacks=fold_callbacks,
        verbose=1,
    )

    best_model = models.load_model(checkpoint_path)
    best_model.save(model_path)

    history_df = pd.read_csv(history_path)
    history_df.insert(0, "fold", fold)
    fold_histories[fold] = history_df

    validation_predictions = best_model.predict(validation_dataset, verbose=0).astype(np.float32)
    if CLIP_NEGATIVE_PREDICTIONS:
        validation_predictions = np.clip(validation_predictions, a_min=0.0, a_max=None)

    validation_indices = validation_rows.index.to_numpy()
    oof_predictions[validation_indices] = validation_predictions

    fold_metrics = compute_metrics_table(y_validation, validation_predictions)
    fold_metrics.insert(0, "fold", fold)
    fold_metrics.insert(0, "feature_set", FEATURE_SET_NAME)
    fold_metrics.insert(0, "model", MODEL_NAME)
    fold_metrics_frames.append(fold_metrics)

    fold_prediction_frame = build_prediction_frame(validation_rows, validation_predictions, fold)
    fold_prediction_frame.to_csv(prediction_path, index=False)
    fold_prediction_frames.append(fold_prediction_frame)

    print(f"Fold {fold} history rows        : {len(history_df)}")
    print(f"Fold {fold} validation rows     : {len(validation_rows)}")
    print(f"Fold {fold} model saved         : {model_path}")
    print(f"Fold {fold} prediction file     : {prediction_path}")
    print(f"Fold {fold} final val_loss      : {history.history['val_loss'][-1]:.6f}")
    print(f"Fold {fold} final val_mae       : {history.history['val_mae'][-1]:.6f}")
    display(fold_metrics)

training_elapsed = time.time() - training_start
print("=" * 60)
print(f"CROSS-VALIDATION COMPLETED IN {training_elapsed:.2f} SECONDS")
print("=" * 60)


TRAINING FOLD 0

Epoch 1/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step - loss: 23.0243 - mae: 23.4993
Epoch 1: val_loss improved from None to 21.21704, saving model to E:\DATAVIDWAN\Image2Biomass\models\cnn_regression\fold0_checkpoint.keras
36/36 ━━━━━━━━━━━━━━━━━━━━ 22s 208ms/step - loss: 23.5454 - mae: 24.0275 - val_loss: 21.2170 - val_mae: 21.7033 - learning_rate: 1.0000e-04
Epoch 2/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step - loss: 22.9276 - mae: 23.4094
Epoch 2: val_loss improved from 21.21704 to 19.43350, saving model to E:\DATAVIDWAN\Image2Biomass\models\cnn_regression\fold0_checkpoint.keras
36/36 ━━━━━━━━━━━━━━━━━━━━ 6s 169ms/step - loss: 21.8016 - mae: 22.2846 - val_loss: 19.4335 - val_mae: 19.9227 - learning_rate: 1.0000e-04
Epoch 3/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step - loss: 19.4284 - mae: 19.9113
Epoch 3: val_loss improved from 19.43350 to 17.55106, saving model to E:\DATAVIDWAN\Image2Biomass\models\cnn_regression\fold0_checkpoint.keras
36/36 ━━━━━━━━━━━━━━━━━━━━ 7s 

,model,feature_set,fold,target,MAE,RMSE,R2
0,EfficientNetB0Regression,images_only,0,Dry_Clover_g,4.834064,10.836774,-0.038396
1,EfficientNetB0Regression,images_only,0,Dry_Dead_g,8.749167,12.873342,-0.161156
2,EfficientNetB0Regression,images_only,0,Dry_Green_g,16.979898,24.472327,-0.052804
3,EfficientNetB0Regression,images_only,0,Dry_Total_g,18.887520,26.475587,-0.097813
4,EfficientNetB0Regression,images_only,0,GDM_g,16.980083,23.622889,-0.042879


TRAINING FOLD 1
Epoch 1/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step - loss: 23.8000 - mae: 24.2642
Epoch 1: val_loss improved from None to 22.70969, saving model to E:\DATAVIDWAN\Image2Biomass\models\cnn_regression\fold1_checkpoint.keras
36/36 ━━━━━━━━━━━━━━━━━━━━ 24s 271ms/step - loss: 23.5893 - mae: 24.0562 - val_loss: 22.7097 - val_mae: 23.1831 - learning_rate: 1.0000e-04
Epoch 2/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step - loss: 22.4368 - mae: 22.9127
Epoch 2: val_loss improved from 22.70969 to 20.89874, saving model to E:\DATAVIDWAN\Image2Biomass\models\cnn_regression\fold1_checkpoint.keras
36/36 ━━━━━━━━━━━━━━━━━━━━ 6s 158ms/step - loss: 21.9190 - mae: 22.3973 - val_loss: 20.8987 - val_mae: 21.3833 - learning_rate: 1.0000e-04
Epoch 3/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step - loss: 20.1320 - mae: 20.6137
Epoch 3: val_loss improved from 20.89874 to 18.95845, saving model to E:\DATAVIDWAN\Image2Biomass\models\cnn_regression\fold1_checkpoint.keras
36/36 ━━━━━━━━━━━━━━━━━━━━ 6s 1

,model,feature_set,fold,target,MAE,RMSE,R2
0,EfficientNetB0Regression,images_only,1,Dry_Clover_g,6.788691,13.384178,-0.099732
1,EfficientNetB0Regression,images_only,1,Dry_Dead_g,10.101488,14.570354,-0.190263
2,EfficientNetB0Regression,images_only,1,Dry_Green_g,16.014286,23.060044,-0.029480
3,EfficientNetB0Regression,images_only,1,Dry_Total_g,21.236572,30.607840,-0.157792
4,EfficientNetB0Regression,images_only,1,GDM_g,16.680008,23.983055,-0.062153


TRAINING FOLD 2
Epoch 1/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - loss: 24.4342 - mae: 24.9135
Epoch 1: val_loss improved from None to 21.87445, saving model to E:\DATAVIDWAN\Image2Biomass\models\cnn_regression\fold2_checkpoint.keras
36/36 ━━━━━━━━━━━━━━━━━━━━ 23s 280ms/step - loss: 23.4469 - mae: 23.9305 - val_loss: 21.8745 - val_mae: 22.3656 - learning_rate: 1.0000e-04
Epoch 2/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step - loss: 21.6131 - mae: 22.1012
Epoch 2: val_loss improved from 21.87445 to 20.25683, saving model to E:\DATAVIDWAN\Image2Biomass\models\cnn_regression\fold2_checkpoint.keras
36/36 ━━━━━━━━━━━━━━━━━━━━ 6s 174ms/step - loss: 21.8749 - mae: 22.3607 - val_loss: 20.2568 - val_mae: 20.7461 - learning_rate: 1.0000e-04
Epoch 3/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step - loss: 20.9520 - mae: 21.4408
Epoch 3: val_loss improved from 20.25683 to 18.56285, saving model to E:\DATAVIDWAN\Image2Biomass\models\cnn_regression\fold2_checkpoint.keras
36/36 ━━━━━━━━━━━━━━━━━━━━ 6s 1

...

TRAINING FOLD 3
Epoch 1/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - loss: 23.3683 - mae: 23.8457
Epoch 1: val_loss improved from None to 23.25764, saving model to E:\DATAVIDWAN\Image2Biomass\models\cnn_regression\fold3_checkpoint.keras
36/36 ━━━━━━━━━━━━━━━━━━━━ 19s 283ms/step - loss: 23.0762 - mae: 23.5545 - val_loss: 23.2576 - val_mae: 23.7429 - learning_rate: 1.0000e-04
Epoch 2/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step - loss: 21.2445 - mae: 21.7264
Epoch 2: val_loss improved from 23.25764 to 21.48452, saving model to E:\DATAVIDWAN\Image2Biomass\models\cnn_regression\fold3_checkpoint.keras
36/36 ━━━━━━━━━━━━━━━━━━━━ 6s 169ms/step - loss: 21.2977 - mae: 21.7814 - val_loss: 21.4845 - val_mae: 21.9743 - learning_rate: 1.0000e-04
Epoch 3/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step - loss: 21.2770 - mae: 21.7657
Epoch 3: val_loss improved from 21.48452 to 19.70225, saving model to E:\DATAVIDWAN\Image2Biomass\models\cnn_regression\fold3_checkpoint.keras
36/36 ━━━━━━━━━━━━━━━━━━━━ 6s 1

,model,feature_set,fold,target,MAE,RMSE,R2
0,EfficientNetB0Regression,images_only,3,Dry_Clover_g,7.924445,14.688876,-0.167816
1,EfficientNetB0Regression,images_only,3,Dry_Dead_g,8.717632,11.974097,-0.070692
2,EfficientNetB0Regression,images_only,3,Dry_Green_g,19.810087,28.144647,-0.043896
3,EfficientNetB0Regression,images_only,3,Dry_Total_g,22.911774,29.636576,-0.124901
4,EfficientNetB0Regression,images_only,3,GDM_g,19.524521,27.426987,-0.137869


TRAINING FOLD 4
Epoch 1/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step - loss: 24.1389 - mae: 24.6029
Epoch 1: val_loss improved from None to 23.75849, saving model to E:\DATAVIDWAN\Image2Biomass\models\cnn_regression\fold4_checkpoint.keras
36/36 ━━━━━━━━━━━━━━━━━━━━ 14s 186ms/step - loss: 23.5243 - mae: 23.9931 - val_loss: 23.7585 - val_mae: 24.2490 - learning_rate: 1.0000e-04
Epoch 2/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step - loss: 22.5995 - mae: 23.0805
Epoch 2: val_loss improved from 23.75849 to 22.28061, saving model to E:\DATAVIDWAN\Image2Biomass\models\cnn_regression\fold4_checkpoint.keras
36/36 ━━━━━━━━━━━━━━━━━━━━ 6s 156ms/step - loss: 22.1276 - mae: 22.6082 - val_loss: 22.2806 - val_mae: 22.7685 - learning_rate: 1.0000e-04
Epoch 3/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step - loss: 21.5082 - mae: 21.9931
Epoch 3: val_loss improved from 22.28061 to 20.65666, saving model to E:\DATAVIDWAN\Image2Biomass\models\cnn_regression\fold4_checkpoint.keras
36/36 ━━━━━━━━━━━━━━━━━━━━ 6s 1

,model,feature_set,fold,target,MAE,RMSE,R2
0,EfficientNetB0Regression,images_only,4,Dry_Clover_g,7.725532,13.713454,-0.176698
1,EfficientNetB0Regression,images_only,4,Dry_Dead_g,8.619841,13.374800,-0.003849
2,EfficientNetB0Regression,images_only,4,Dry_Green_g,17.649481,28.115576,-0.093825
3,EfficientNetB0Regression,images_only,4,Dry_Total_g,21.867371,34.901579,-0.196509
4,EfficientNetB0Regression,images_only,4,GDM_g,18.798693,29.552449,-0.204584


CROSS-VALIDATION COMPLETED IN 416.55 SECONDS


In [11]:
fold_metrics = pd.concat(fold_metrics_frames, ignore_index=True)
oof_predictions_validated = validate_prediction_array(oof_predictions, expected_rows=len(train_df))
oof_predictions_df = build_prediction_frame(train_df, oof_predictions, fold_value="OOF")
oof_predictions_df.insert(0, "model", MODEL_NAME)
oof_predictions_df.insert(1, "feature_set", FEATURE_SET_NAME)

print("=" * 60)
print("FOLD METRICS PREVIEW")
print("=" * 60)
print(fold_metrics.head(10))

print("=" * 60)
print("OOF PREDICTION VALIDATION")
print("=" * 60)
for key, value in oof_predictions_validated.items():
    print(f"{key:>24} : {value}")


FOLD METRICS PREVIEW
                      model  feature_set  fold        target        MAE  \
0  EfficientNetB0Regression  images_only     0  Dry_Clover_g   4.834064   
1  EfficientNetB0Regression  images_only     0    Dry_Dead_g   8.749167   
2  EfficientNetB0Regression  images_only     0   Dry_Green_g  16.979898   
3  EfficientNetB0Regression  images_only     0   Dry_Total_g  18.887520   
4  EfficientNetB0Regression  images_only     0         GDM_g  16.980083   
5  EfficientNetB0Regression  images_only     1  Dry_Clover_g   6.788691   
6  EfficientNetB0Regression  images_only     1    Dry_Dead_g  10.101488   
7  EfficientNetB0Regression  images_only     1   Dry_Green_g  16.014286   
8  EfficientNetB0Regression  images_only     1   Dry_Total_g  21.236572   
9  EfficientNetB0Regression  images_only     1         GDM_g  16.680008   

        RMSE        R2  
0  10.836774 -0.038396  
1  12.873342 -0.161156  
2  24.472327 -0.052804  
3  26.475587 -0.097813  
4  23.622889 -0.042879  
5  

In [12]:
saved_model_paths = [CNN_REGRESSION_MODEL_DIR / f"fold{fold}.keras" for fold in range(N_FOLDS)]
saved_history_paths = [CNN_REGRESSION_MODEL_DIR / f"fold{fold}_history.csv" for fold in range(N_FOLDS)]
saved_prediction_paths = [PREDICTIONS_DIR / f"07_fold{fold}_oof_predictions.csv" for fold in range(N_FOLDS)]

assert all(path.exists() for path in saved_model_paths), "Not all fold models were saved."
assert all(path.exists() for path in saved_history_paths), "Not all fold histories were saved."
assert all(path.exists() for path in saved_prediction_paths), "Not all fold prediction files were saved."
assert np.isfinite(oof_predictions).all(), "OOF predictions contain NaN or Inf values."
assert len(fold_metrics) == N_FOLDS * len(TARGET_COLUMNS), "Unexpected fold metrics row count."

print("All fold models, histories, and prediction files exist.")
print(f"Saved models      : {len(saved_model_paths)}")
print(f"Saved histories   : {len(saved_history_paths)}")
print(f"Saved prediction CSVs : {len(saved_prediction_paths)}")


All fold models, histories, and prediction files exist.
Saved models      : 5
Saved histories   : 5
Saved prediction CSVs : 5


## Aggregate metrics

The tables below summarize performance in two complementary ways:

- `07_cv_summary.csv` captures the mean and standard deviation across folds for every target.
- `07_per_target_metrics.csv` captures the final OOF metrics for the full set of predictions.
- `07_model_ranking.csv` collapses those results into a single model-level score for this experiment.

In [13]:
cv_summary = summarize_fold_metrics(fold_metrics)
per_target_metrics = compute_metrics_table(train_df[TARGET_COLUMNS].to_numpy(dtype=np.float32), oof_predictions)
per_target_metrics.insert(0, "model", MODEL_NAME)
per_target_metrics.insert(1, "feature_set", FEATURE_SET_NAME)
model_ranking = build_model_ranking(cv_summary)

oof_overall_mae = float(mean_absolute_error(train_df[TARGET_COLUMNS].to_numpy(dtype=np.float32), oof_predictions))
oof_overall_rmse = float(np.sqrt(mean_squared_error(train_df[TARGET_COLUMNS].to_numpy(dtype=np.float32), oof_predictions)))
oof_overall_r2 = float(r2_score(train_df[TARGET_COLUMNS].to_numpy(dtype=np.float32), oof_predictions, multioutput="uniform_average"))

overall_summary = pd.DataFrame(
    [
        {
            "model": MODEL_NAME,
            "feature_set": FEATURE_SET_NAME,
            "OOF_MAE": oof_overall_mae,
            "OOF_RMSE": oof_overall_rmse,
            "OOF_R2": oof_overall_r2,
            "folds": N_FOLDS,
            "samples": len(train_df),
        }
    ]
)

print("=" * 60)
print("CV SUMMARY")
print("=" * 60)
display(cv_summary)
print("=" * 60)
print("PER-TARGET OOF METRICS")
print("=" * 60)
display(per_target_metrics)
print("=" * 60)
print("MODEL RANKING")
print("=" * 60)
display(model_ranking)
print("=" * 60)
print("OVERALL SUMMARY")
print("=" * 60)
display(overall_summary)


CV SUMMARY


,model,feature_set,target,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std
0,EfficientNetB0Regression,images_only,Dry_Clover_g,6.604601,1.313111,12.680692,1.775266,-0.122686,0.056239
1,EfficientNetB0Regression,images_only,Dry_Dead_g,8.862556,0.736924,12.890323,1.163301,-0.097311,0.076661
2,EfficientNetB0Regression,images_only,Dry_Green_g,17.686540,1.404644,26.005923,2.242114,-0.055603,0.023948
3,EfficientNetB0Regression,images_only,GDM_g,17.975939,1.199221,26.080740,2.468568,-0.102518,0.067552
4,EfficientNetB0Regression,images_only,Dry_Total_g,20.921752,1.625639,29.797858,3.304284,-0.144973,0.036931


PER-TARGET OOF METRICS


,model,feature_set,target,MAE,RMSE,R2
0,EfficientNetB0Regression,images_only,Dry_Clover_g,6.600158,12.776423,-0.114787
1,EfficientNetB0Regression,images_only,Dry_Dead_g,8.865709,12.936963,-0.091186
2,EfficientNetB0Regression,images_only,Dry_Green_g,17.679876,26.070777,-0.056371
3,EfficientNetB0Regression,images_only,Dry_Total_g,20.916937,29.936788,-0.147648
4,EfficientNetB0Regression,images_only,GDM_g,17.969519,26.161353,-0.103802


MODEL RANKING


,rank,model,feature_set,mean_MAE,mean_RMSE,mean_R2
0,1,EfficientNetB0Regression,images_only,14.410278,21.491107,-0.104618


OVERALL SUMMARY


,model,feature_set,OOF_MAE,OOF_RMSE,OOF_R2,folds,samples
0,EfficientNetB0Regression,images_only,14.406439,22.763632,-0.102759,5,357


In [14]:
fold_metrics_path = REPORTS_DIR / "07_fold_metrics.csv"
cv_summary_path = REPORTS_DIR / "07_cv_summary.csv"
model_ranking_path = REPORTS_DIR / "07_model_ranking.csv"
per_target_metrics_path = REPORTS_DIR / "07_per_target_metrics.csv"
oof_predictions_path = PREDICTIONS_DIR / "07_oof_predictions.csv"

fold_metrics.to_csv(fold_metrics_path, index=False)
cv_summary.to_csv(cv_summary_path, index=False)
model_ranking.to_csv(model_ranking_path, index=False)
per_target_metrics.to_csv(per_target_metrics_path, index=False)
oof_predictions_df.to_csv(oof_predictions_path, index=False)

print("=" * 60)
print("SAVED REPORTS")
print("=" * 60)
print(f"{fold_metrics_path}")
print(f"{cv_summary_path}")
print(f"{model_ranking_path}")
print(f"{per_target_metrics_path}")
print(f"{oof_predictions_path}")


SAVED REPORTS
E:\DATAVIDWAN\Image2Biomass\outputs\reports\07_fold_metrics.csv
E:\DATAVIDWAN\Image2Biomass\outputs\reports\07_cv_summary.csv
E:\DATAVIDWAN\Image2Biomass\outputs\reports\07_model_ranking.csv
E:\DATAVIDWAN\Image2Biomass\outputs\reports\07_per_target_metrics.csv
E:\DATAVIDWAN\Image2Biomass\outputs\predictions\07_oof_predictions.csv


## Aggregated snapshot

This checkpoint makes the fold-level and OOF summaries easy to inspect before the visual diagnostics are generated.

In [15]:
display(cv_summary)
display(per_target_metrics)
display(model_ranking)


,model,feature_set,target,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std
0,EfficientNetB0Regression,images_only,Dry_Clover_g,6.604601,1.313111,12.680692,1.775266,-0.122686,0.056239
1,EfficientNetB0Regression,images_only,Dry_Dead_g,8.862556,0.736924,12.890323,1.163301,-0.097311,0.076661
2,EfficientNetB0Regression,images_only,Dry_Green_g,17.686540,1.404644,26.005923,2.242114,-0.055603,0.023948
3,EfficientNetB0Regression,images_only,GDM_g,17.975939,1.199221,26.080740,2.468568,-0.102518,0.067552
4,EfficientNetB0Regression,images_only,Dry_Total_g,20.921752,1.625639,29.797858,3.304284,-0.144973,0.036931


,model,feature_set,target,MAE,RMSE,R2
0,EfficientNetB0Regression,images_only,Dry_Clover_g,6.600158,12.776423,-0.114787
1,EfficientNetB0Regression,images_only,Dry_Dead_g,8.865709,12.936963,-0.091186
2,EfficientNetB0Regression,images_only,Dry_Green_g,17.679876,26.070777,-0.056371
3,EfficientNetB0Regression,images_only,Dry_Total_g,20.916937,29.936788,-0.147648
4,EfficientNetB0Regression,images_only,GDM_g,17.969519,26.161353,-0.103802


,rank,model,feature_set,mean_MAE,mean_RMSE,mean_R2
0,1,EfficientNetB0Regression,images_only,14.410278,21.491107,-0.104618


In [16]:
fold_target_means = train_df.groupby("fold")[TARGET_COLUMNS].mean().round(3)
display(fold_target_means)


,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g
fold,,,,,
0,4.474,12.563,26.635,43.668,31.109
1,6.745,13.711,25.426,45.882,32.171
2,6.010,11.482,26.685,44.177,32.695
3,8.189,11.749,26.830,46.768,35.019
4,7.859,10.688,27.565,46.112,35.424


## Saved artifact inventory

The notebook now has all persisted outputs in place: fold models, fold histories, OOF prediction files, reports, and figures.

In [18]:
artifact_inventory = pd.DataFrame(
    [
        {
            "artifact_type": "model",
            "count": len(saved_model_paths),
            "location": str(CNN_REGRESSION_MODEL_DIR),
        },
        {
            "artifact_type": "history",
            "count": len(saved_history_paths),
            "location": str(CNN_REGRESSION_MODEL_DIR),
        },
        {
            "artifact_type": "fold_predictions",
            "count": len(saved_prediction_paths),
            "location": str(PREDICTIONS_DIR),
        },
        {
            "artifact_type": "reports",
            "count": "Will be generated later",
            "location": str(REPORTS_DIR),
        },
        {
            "artifact_type": "figures",
            "count": "Will be generated later",
            "location": str(DEEP_LEARNING_FIGURES_DIR),
        },
    ]
)

display(artifact_inventory)

,artifact_type,count,location
0,model,5,E:\DATAVIDWAN\Image2Biomass\models\cnn_regression
1,history,5,E:\DATAVIDWAN\Image2Biomass\models\cnn_regression
2,fold_predictions,5,E:\DATAVIDWAN\Image2Biomass\outputs\predictions
3,reports,Will be generated later,E:\DATAVIDWAN\Image2Biomass\outputs\reports
4,figures,Will be generated later,E:\DATAVIDWAN\Image2Biomass\outputs\figures\de...


## Learning curves

The next cells visualize the fold histories and the OOF prediction diagnostics produced by this run.

In [19]:
history_figure_path = DEEP_LEARNING_FIGURES_DIR / "07_learning_curves.png"
save_history_figure(fold_histories, history_figure_path)
print(f"Saved: {history_figure_path}")


Saved: E:\DATAVIDWAN\Image2Biomass\outputs\figures\deep_learning\07_learning_curves.png


In [20]:
scatter_figure_path = DEEP_LEARNING_FIGURES_DIR / "07_actual_vs_predicted.png"
save_prediction_scatter(oof_predictions_df, scatter_figure_path)
print(f"Saved: {scatter_figure_path}")


Saved: E:\DATAVIDWAN\Image2Biomass\outputs\figures\deep_learning\07_actual_vs_predicted.png


In [21]:
residual_plot_path = DEEP_LEARNING_FIGURES_DIR / "07_residual_plots.png"
save_residual_plots(oof_predictions_df, residual_plot_path)
print(f"Saved: {residual_plot_path}")


Saved: E:\DATAVIDWAN\Image2Biomass\outputs\figures\deep_learning\07_residual_plots.png


In [22]:
residual_histogram_path = DEEP_LEARNING_FIGURES_DIR / "07_residual_histogram.png"
save_residual_histogram(oof_predictions_df, residual_histogram_path)
print(f"Saved: {residual_histogram_path}")


Saved: E:\DATAVIDWAN\Image2Biomass\outputs\figures\deep_learning\07_residual_histogram.png


In [23]:
prediction_distribution_path = DEEP_LEARNING_FIGURES_DIR / "07_prediction_distribution.png"
save_prediction_distribution(oof_predictions_df, prediction_distribution_path)
print(f"Saved: {prediction_distribution_path}")


Saved: E:\DATAVIDWAN\Image2Biomass\outputs\figures\deep_learning\07_prediction_distribution.png


In [24]:
rmse_barplot_path = DEEP_LEARNING_FIGURES_DIR / "07_per_target_rmse.png"
mae_barplot_path = DEEP_LEARNING_FIGURES_DIR / "07_per_target_mae.png"
save_target_barplot(per_target_metrics, "RMSE", rmse_barplot_path)
save_target_barplot(per_target_metrics, "MAE", mae_barplot_path)
print(f"Saved: {rmse_barplot_path}")
print(f"Saved: {mae_barplot_path}")


Saved: E:\DATAVIDWAN\Image2Biomass\outputs\figures\deep_learning\07_per_target_rmse.png
Saved: E:\DATAVIDWAN\Image2Biomass\outputs\figures\deep_learning\07_per_target_mae.png


## Final integrity validation

The final audit checks the same categories used throughout the earlier notebooks:

- schema and sample-count consistency;
- fold coverage;
- numerical validity of predictions;
- expected artifact files on disk;
- expected report files on disk;
- expected figure files on disk.

In [25]:
expected_report_paths = [
    fold_metrics_path,
    cv_summary_path,
    model_ranking_path,
    per_target_metrics_path,
    oof_predictions_path,
]
expected_figure_paths = [
    history_figure_path,
    scatter_figure_path,
    residual_plot_path,
    residual_histogram_path,
    prediction_distribution_path,
    rmse_barplot_path,
    mae_barplot_path,
]
expected_model_paths = saved_model_paths
expected_history_paths = saved_history_paths
expected_prediction_paths = saved_prediction_paths

integrity_checks = [
    {
        "check": "sample_count",
        "expected": len(train_df),
        "actual": len(oof_predictions_df),
        "status": "PASS" if len(train_df) == len(oof_predictions_df) else "FAIL",
    },
    {
        "check": "target_count",
        "expected": len(TARGET_COLUMNS),
        "actual": len([column for column in oof_predictions_df.columns if column.startswith("actual_")]),
        "status": "PASS" if len([column for column in oof_predictions_df.columns if column.startswith("actual_")]) == len(TARGET_COLUMNS) else "FAIL",
    },
    {
        "check": "fold_count",
        "expected": N_FOLDS,
        "actual": train_df["fold"].nunique(),
        "status": "PASS" if train_df["fold"].nunique() == N_FOLDS else "FAIL",
    },
    {
        "check": "prediction_nan_count",
        "expected": 0,
        "actual": int(np.isnan(oof_predictions).sum()),
        "status": "PASS" if int(np.isnan(oof_predictions).sum()) == 0 else "FAIL",
    },
    {
        "check": "prediction_inf_count",
        "expected": 0,
        "actual": int(np.isinf(oof_predictions).sum()),
        "status": "PASS" if int(np.isinf(oof_predictions).sum()) == 0 else "FAIL",
    },
    {
        "check": "saved_model_count",
        "expected": N_FOLDS,
        "actual": sum(path.exists() for path in expected_model_paths),
        "status": "PASS" if sum(path.exists() for path in expected_model_paths) == N_FOLDS else "FAIL",
    },
    {
        "check": "saved_history_count",
        "expected": N_FOLDS,
        "actual": sum(path.exists() for path in expected_history_paths),
        "status": "PASS" if sum(path.exists() for path in expected_history_paths) == N_FOLDS else "FAIL",
    },
    {
        "check": "saved_fold_prediction_count",
        "expected": N_FOLDS,
        "actual": sum(path.exists() for path in expected_prediction_paths),
        "status": "PASS" if sum(path.exists() for path in expected_prediction_paths) == N_FOLDS else "FAIL",
    },
    {
        "check": "report_file_count",
        "expected": len(expected_report_paths),
        "actual": sum(path.exists() for path in expected_report_paths),
        "status": "PASS" if sum(path.exists() for path in expected_report_paths) == len(expected_report_paths) else "FAIL",
    },
    {
        "check": "figure_file_count",
        "expected": len(expected_figure_paths),
        "actual": sum(path.exists() for path in expected_figure_paths),
        "status": "PASS" if sum(path.exists() for path in expected_figure_paths) == len(expected_figure_paths) else "FAIL",
    },
]

final_integrity_report = pd.DataFrame(integrity_checks)
final_integrity_report_path = REPORTS_DIR / "07_final_integrity_report.csv"
final_integrity_report.to_csv(final_integrity_report_path, index=False)

print("=" * 60)
print("FINAL INTEGRITY REPORT")
print("=" * 60)
display(final_integrity_report)
print(f"Saved: {final_integrity_report_path}")


FINAL INTEGRITY REPORT


,check,expected,actual,status
0,sample_count,357,357,PASS
1,target_count,5,5,PASS
2,fold_count,5,5,PASS
3,prediction_nan_count,0,0,PASS
4,prediction_inf_count,0,0,PASS
5,saved_model_count,5,5,PASS
6,saved_history_count,5,5,PASS
7,saved_fold_prediction_count,5,5,PASS
8,report_file_count,5,5,PASS
9,figure_file_count,7,7,PASS


Saved: E:\DATAVIDWAN\Image2Biomass\outputs\reports\07_final_integrity_report.csv


In [26]:
assert (final_integrity_report["status"] == "PASS").all(), "One or more final integrity checks failed."
print("All final integrity checks passed.")
print("=" * 60)
print("DEEP LEARNING REGRESSION SUMMARY")
print("=" * 60)
print(overall_summary.to_string(index=False))
print("=" * 60)
print("Project artifacts are ready for downstream comparison with Notebooks 04 to 06.")


All final integrity checks passed.
DEEP LEARNING REGRESSION SUMMARY
                   model feature_set   OOF_MAE  OOF_RMSE    OOF_R2  folds  samples
EfficientNetB0Regression images_only 14.406439 22.763632 -0.102759      5      357
Project artifacts are ready for downstream comparison with Notebooks 04 to 06.


## Final Conclusions

Notebook 07 evaluated an end-to-end EfficientNetB0 regression model using five-fold cross-validation on the CSIRO Image2Biomass dataset.

The model converged consistently across all folds and produced stable training and validation loss curves, demonstrating that the optimization procedure was effective.

However, the final predictive performance remained below the advanced classical machine learning models developed in Notebook 06. The EfficientNetB0 model achieved an average RMSE of approximately **21.49 g**, average MAE of **14.41 g**, and mean R² of **−0.10**, indicating limited generalization performance on this relatively small dataset.

The primary limitation observed throughout the evaluation is the tendency of the network to predict values within a restricted range, reducing its ability to estimate high-biomass samples accurately.

Compared with the classical pipeline, the results demonstrate that combining pretrained CNN embeddings with metadata and handcrafted visual descriptors provides substantially better predictive performance than direct end-to-end image regression under the available training data.

The trained CNN models, prediction artifacts, learning curves, and diagnostic visualizations nevertheless provide a valuable deep learning benchmark for future work involving larger datasets, stronger augmentation strategies, or multimodal fusion approaches.